# RAG Example

In this example we build and export a Chroma Database that can be used on a local LLM execution for RAG.
The database is created against a Github repo passed as parameter.

This notebook can run locally (make it sure to have installed the necessary dependencies) or on KFP. When running on KFP, it can be scheduled to regularly run against a github repo, generating fresh vector databases.

In [ ]:
import shutil
import hashlib
import sentence_transformers
from langchain_community.document_loaders import DirectoryLoader,TextLoader
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from git import Repo
from langchain_chroma import Chroma

### Pipeline parameters
Using these parameters we can control which files will be read from the remote repo and loaded into the vector database

In [ ]:
files_pattern = "**/*.py,**/*.ts,**/*.tsx,**/*.md"
repo = "kubeflow/kale"
git_url = "https://github.com"
embeddings_model = "all-MiniLM-L6-v2"

### Clone repo and retrieving documents
We clone the repo provided by the user and load the files into documents that will be used to build the database. This is a step that is not cached so we can always get the latest changes from the target repo.

In [ ]:
# Clone the repo and save the zipped content
repo_dir = '/tmp/repo'
repo_url = f'{git_url}/{repo}'
shutil.rmtree(repo_dir, ignore_errors=True)
Repo.clone_from(repo_url, repo_dir)
loader = DirectoryLoader(repo_dir, glob=files_pattern.split(","),loader_cls=TextLoader,use_multithreading=True)
docs = loader.load()
if len(docs) == 0:
    raise Exception("No documents to build the vector database")
print(f'Finished reading {len(docs)} docs from repo {repo_url}')

### Loading embeddings model

The embeddings model is downloaded and loaded. Notice that we ensure that this step is cached so we don't have to re-download the model for each run

In [ ]:
embeddings = HuggingFaceEmbeddings(model_name=embeddings_model)

### Create Vector DB
We use the embeddings and the documents from the github repo to build the vector database, zip and store the zip file contents in a variable

In [ ]:
db_dir = "/tmp/chroma_db"
Chroma.from_documents(
    documents=docs, 
    embedding=embeddings, 
    persist_directory=db_dir
)
zip_file = shutil.make_archive('/tmp/chroma_db', 'zip', db_dir)
with open(zip_file, 'rb') as f:
    chroma_db = f.read()
print(f'File succesfully created and stored in {db_dir}')

### Check the vector DB
On this step we check the produced database and Kale will have to create the chroma_db output artifact to be passed to this, so we can download it from KFP

In [ ]:
md5_hash = hashlib.md5(chroma_db).hexdigest()
print(f'Vector DB created sucessfully. MD5 Hash: {md5_hash}')

### Verify the Vector DB
With this code we verify if the file can be loaded correcly from the file system and it is skipped for the pipeline execution. Notice that this can run on a different computer using the artifact downloaded from KFP

In [ ]:
vector_db = Chroma(
  embedding_function=embeddings,
  persist_directory=db_dir
)
vector_db.similarity_search("What is Kale?", k=1)